In [1]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import numpy as np
import pandas as pd
from keras.models import Model, Sequential
from keras.layers import Input, Dense, Conv1D, Flatten, MaxPooling1D, Concatenate, Dropout
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

In [2]:
%matplotlib inline

In [3]:
seed_value = 0
os.environ['PYTHONHASHSEED'] = str(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

In [4]:
value_df = 10000
star_check = pd.read_csv("./Datasets/updated_database_exoplanet.csv")
star_check = star_check.drop(['tid'],axis=1)
star_check_y = star_check[['confirmed_planet']]
star_check = star_check.reset_index().drop('index',axis=1)
star_check = star_check.apply(lambda row: row.fillna(0), axis=1)

In [7]:
avg_val = int(star_check.value_counts(star_check['confirmed_planet']).mean())
balanced_star_check = pd.DataFrame()
class_counts = star_check['confirmed_planet'].value_counts()
for i in range(0,2):
    if class_counts.loc[class_counts.index == i].iloc[0] > avg_val:
        balanced_star_check = pd.concat([balanced_star_check, star_check[star_check['confirmed_planet'] == i].sample(avg_val)])
    else:
        balanced_star_check = pd.concat([balanced_star_check, star_check[star_check['confirmed_planet'] == i].sample(avg_val, replace=True)])

In [8]:
star_check = balanced_star_check.copy()

In [9]:
scaler = MinMaxScaler()
star_check[['Teff','logg','MH','rad','mass','rho','lum','Tmag','ra','dec','plx']] = scaler.fit_transform(star_check[['Teff','logg','MH','rad','mass','rho','lum','Tmag','ra','dec','plx']])

In [10]:
X_train, X_test, y_train, y_test = train_test_split(star_check.drop('confirmed_planet',axis=1),star_check[['confirmed_planet']], test_size=0.1, random_state=42)

In [11]:
X_train_flux = X_train.drop(['Teff','logg','MH','rad','mass','rho','lum','Tmag','ra','dec','plx'],axis=1)
X_train_params = X_train[['Teff','logg','MH','rad','mass','rho','lum','Tmag','ra','dec','plx']]
X_test_flux = X_test.drop(['Teff','logg','MH','rad','mass','rho','lum','Tmag','ra','dec','plx'],axis=1)
X_test_params = X_test[['Teff','logg','MH','rad','mass','rho','lum','Tmag','ra','dec','plx']]

In [12]:
input_flux = Input(shape=(10000,1))
x = Conv1D(filters=64, kernel_size=3, activation='relu')(input_flux)
x = MaxPooling1D(pool_size=2)(x)
x = Conv1D(filters=64, kernel_size=5, activation='relu')(x)
x = MaxPooling1D(pool_size=2)(x)
x = Conv1D(filters=64, kernel_size=5, activation='relu')(x)
x = MaxPooling1D(pool_size=2)(x)
x = Flatten()(x)
x = Dropout(0.75)(x)
model_flux = Model(inputs=input_flux, outputs=x)

input_params = Input(shape=(11,))
y = Dense(128, activation='relu')(input_params)
model_params = Model(inputs=input_params, outputs=y)

combined = Concatenate()([model_flux.output, model_params.output])
z = Dropout(0.5)(combined)
z = Dense(128, activation='relu')(z)
final_output = Dense(1, activation='sigmoid')(z)

model = Model(inputs=[model_flux.input, model_params.input], outputs=final_output)

2024-04-04 23:25:17.890166: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2024-04-04 23:25:17.890258: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2024-04-04 23:25:17.890272: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2024-04-04 23:25:17.890332: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2024-04-04 23:25:17.890808: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [13]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',  
    patience=10,        
    mode='min'          
)

class MyThresholdCallback(tf.keras.callbacks.Callback):
    def __init__(self, threshold):
        super(MyThresholdCallback, self).__init__()
        self.num_cross = 0
        self.threshold = threshold

    def on_epoch_end(self, epoch, logs=None): 
        val_acc = logs["val_accuracy"]
        acc = logs['accuracy']
        if val_acc >= self.threshold and acc >= self.threshold:
            self.num_cross += 1
            if self.num_cross >= 3:
                self.model.stop_training = True

my_callback = MyThresholdCallback(threshold=0.91)

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# history = model.fit([X_train_flux, X_train_params], y_train, epochs=100, batch_size=32, validation_split=0.2, callbacks=[my_callback, early_stopping])

In [14]:
model.load_weights('./Models/model_weights_v3.h5')

In [15]:
loss, accuracy = model.evaluate([X_test_flux,X_test_params],y_test)

2024-04-04 23:25:18.982983: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.9103 - loss: 0.2385


#### Loss Graph

![alt text](loss_graph.png)

#### Accuracy Graph

![alt text](acc_graph.png)